# 3D Monte Carlo Simulation - Neural Network Surrogate Model

This notebook trains a neural network to predict the outcome of 3D photon transport simulations.

**Inputs:**
- `tau_tot`: Optical depth
- `omega`: Single scattering albedo
- `g`: Asymmetry parameter (Henyey-Greenstein)

**Outputs:**
- `escape_fraction_top`: Fraction of photons escaping the top
- `escape_fraction_bottom`: Fraction of photons escaping the bottom
- `absorbed_fraction`: Fraction of photons absorbed

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# Load dataset
data = pd.read_csv('monte_carlo_results_3d.csv')
print(f"Loaded {len(data)} samples")
data.head()

Loaded 300 samples


,tau_tot,omega,g,escape_fraction_top,escape_fraction_bottom,absorbed_fraction
0,0.1,0.00,0.0,0.7304,0.0000,0.2696
1,0.1,0.11,0.0,0.7452,0.0104,0.2444
2,0.1,0.22,0.0,0.7514,0.0290,0.2196
3,0.1,0.33,0.0,0.7722,0.0400,0.1878
4,0.1,0.44,0.0,0.7728,0.0522,0.1750


In [3]:
# Parse into inputs (X) and outputs (Y)
X = data[['tau_tot', 'omega', 'g']].values
Y = data[['escape_fraction_top', 'escape_fraction_bottom', 'absorbed_fraction']].values

print("Input shape:", X.shape)
print("Output shape:", Y.shape)

Input shape: (300, 3)
Output shape: (300, 3)


In [4]:
# Create training and testing datasets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [5]:
# Normalize the inputs
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

In [6]:
# Build model with 3 inputs and 3 outputs
class RegressionNet3D(nn.Module): 
    def __init__(self):
        super(RegressionNet3D, self).__init__()
        # Input: 3 (tau, omega, g)
        self.fc1 = nn.Linear(3, 64)
        self.fc2 = nn.Linear(64, 64)
        # Output: 3 (escape_top, escape_bottom, absorbed)
        self.fc3 = nn.Linear(64, 3)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
# Set model, error function, and learning rate
model = RegressionNet3D()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
# Convert numpy arrays to pytorch tensors
X_train_tensor = torch.FloatTensor(X_train_s)
Y_train_tensor = torch.FloatTensor(Y_train)
X_test_tensor = torch.FloatTensor(X_test_s)
Y_test_tensor = torch.FloatTensor(Y_test)

In [9]:
# Create DataLoaders
val_size = int(0.1 * len(X_train_tensor))
train_size = len(X_train_tensor) - val_size

train_dataset = TensorDataset(X_train_tensor[:train_size], Y_train_tensor[:train_size])
val_dataset = TensorDataset(X_train_tensor[train_size:], Y_train_tensor[train_size:])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [10]:
# Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    
    # Train
    for X_batch, Y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, Y_batch)
        loss.backward()
        optimizer.step()
    
    # Validation
    model.eval()
    val_loss = 0
    val_mae = 0
    with torch.no_grad():
        for X_val, Y_val in val_loader:
            outputs = model(X_val)
            val_loss += criterion(outputs, Y_val).item()
            val_mae += torch.mean(torch.abs(outputs - Y_val)).item()
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Val Loss: {val_loss/len(val_loader):.4f}, Val MAE: {val_mae/len(val_loader):.4f}')

Epoch 20/200, Val Loss: 0.0120, Val MAE: 0.0760
Epoch 40/200, Val Loss: 0.0050, Val MAE: 0.0438
Epoch 60/200, Val Loss: 0.0041, Val MAE: 0.0368
Epoch 80/200, Val Loss: 0.0039, Val MAE: 0.0336
Epoch 100/200, Val Loss: 0.0036, Val MAE: 0.0337
Epoch 120/200, Val Loss: 0.0035, Val MAE: 0.0311
Epoch 140/200, Val Loss: 0.0032, Val MAE: 0.0291
Epoch 160/200, Val Loss: 0.0034, Val MAE: 0.0309
Epoch 180/200, Val Loss: 0.0030, Val MAE: 0.0274
Epoch 200/200, Val Loss: 0.0030, Val MAE: 0.0271


In [11]:
# Evaluation on Test Set
model.eval()
with torch.no_grad():
    Y_pred = model(X_test_tensor).numpy()
    test_mae = torch.mean(torch.abs(model(X_test_tensor) - Y_test_tensor)).item()

print('Test MAE:', test_mae)

# Compute percent errors (adding epsilon to avoid division by zero)
percent_errors = 100.0 * np.abs((Y_test - Y_pred) / (Y_test + 1e-12))

print('Median percent error (Escape Top):', np.median(percent_errors[:,0]))
print('Median percent error (Escape Bottom):', np.median(percent_errors[:,1]))
print('Median percent error (Absorbed):', np.median(percent_errors[:,2]))

Test MAE: 0.01685633696615696
Median percent error (Escape Top): 5.008630036930017
Median percent error (Escape Bottom): 11.885305467185912
Median percent error (Absorbed): 2.5967128383300966
